# Surviving a kill: partial manifest recovery

The durability claim is narrow but load-bearing: **the JSON Lines manifest survives a mid-sweep `Ctrl-C`**, the partial DataFrame is loadable from disk, and the CLI inspector reports the partial state. This notebook walks through that exact flow end-to-end:

1. Launch `gmat-sweep run` as a subprocess against a small mission.
2. Poll the manifest until a few runs have completed.
3. Send the subprocess `SIGINT` (the same signal `Ctrl-C` raises).
4. Inspect the partial manifest with `gmat-sweep show`.
5. Reload the manifest in-process and aggregate the partial DataFrame from the Parquet files on disk.

**Programmatic resume — `Sweep.from_manifest(...).resume()` — lands in v0.2.** Until then, the surface this notebook exercises is durability and inspection: a killed sweep leaves a parseable manifest and queryable per-run artefacts, but you re-issue the original `sweep()` call to re-run the missing cells.

**Prerequisites.** A local GMAT install (R2026a is the primary development target). This notebook does not depend on the `[examples]` extra (no plots).

**Platform note.** The `SIGINT`-via-subprocess approach used below is POSIX-only. On Windows, the same recovery flow works after a real `Ctrl-C` in an interactive `gmat-sweep run` session — but reproducing the kill from a notebook cell needs different signalling primitives.

## Set up the run

Resolve the GMAT install once and confirm the small mission script that ships next to this notebook is where we expect it. The script propagates for 60 seconds with point-mass Earth — every per-run cost is sub-second so the kill-and-inspect demo finishes inside the smoke step's wall-clock budget.

In [ ]:
import signal
import subprocess
import tempfile
import time
from pathlib import Path

from gmat_run import locate_gmat

from gmat_sweep import Manifest, lazy_multiindex

install = locate_gmat()
script_path = Path("leo_short.script").resolve()

print(f"GMAT version: {install.version}")
print(f"Script:       {script_path.name}")
print(f"Exists:       {script_path.exists()}")

## Launch the sweep as a subprocess

We dispatch 20 runs with `--workers 1` so completions are serialised — that makes the polling loop below predictable. Each completed run appends one fsync'd line to `manifest.jsonl`.

In [ ]:
tmpdir = tempfile.TemporaryDirectory(prefix="killed-sweep-")
out_dir = Path(tmpdir.name)
manifest_path = out_dir / "manifest.jsonl"

proc = subprocess.Popen(
    [
        "gmat-sweep",
        "run",
        "--grid",
        "Sat.SMA=7000:8000:20",
        "--workers",
        "1",
        "--out",
        str(out_dir),
        str(script_path),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(f"Spawned gmat-sweep pid={proc.pid}, output dir = {out_dir}")

## Wait for a few runs to finish, then SIGINT

Poll the manifest until at least 5 runs have completed. Each fsync'd append is durable, so a parse mid-poll always sees a consistent prefix.

Once the threshold is reached we send the child `SIGINT` — the same signal an interactive `Ctrl-C` would raise — and wait for it to exit. The orchestrator does not catch `KeyboardInterrupt`, so the subprocess exits with a non-zero code; that is the expected outcome.

In [ ]:
TARGET_ENTRIES = 5
DEADLINE_S = 60.0
deadline = time.monotonic() + DEADLINE_S


def _entries_on_disk(path: Path) -> int:
    if not path.exists():
        return 0
    # Header line + one per-run line. Subtract 1 for the header.
    return max(0, sum(1 for _ in path.open("r", encoding="utf-8")) - 1)


while time.monotonic() < deadline:
    if proc.poll() is not None:
        raise RuntimeError(
            f"gmat-sweep exited before SIGINT could be sent (rc={proc.returncode}); "
            f"stderr: {proc.stderr.read().decode(errors='replace') if proc.stderr else ''}"
        )
    if _entries_on_disk(manifest_path) >= TARGET_ENTRIES:
        break
    time.sleep(0.1)
else:
    proc.kill()
    raise RuntimeError(
        f"gmat-sweep did not append {TARGET_ENTRIES} manifest entries within {DEADLINE_S} s"
    )

n_before_kill = _entries_on_disk(manifest_path)
proc.send_signal(signal.SIGINT)
rc = proc.wait(timeout=30)
n_after_kill = _entries_on_disk(manifest_path)

print(f"Subprocess exited with rc={rc}")
print(f"Manifest entries at kill:  {n_before_kill}")
print(f"Manifest entries on disk:  {n_after_kill}")

## Inspect the partial manifest with `gmat-sweep show`

The CLI's `show` subcommand prints a one-line summary: counts per status bucket, total wall-clock duration, output directory. It loads the manifest with the same [`Manifest.load`][gmat_sweep.Manifest.load] used in-process, so a torn final line (if the kill landed mid-write) is tolerated — at most one entry is lost.

In [ ]:
show = subprocess.run(
    ["gmat-sweep", "show", str(manifest_path)],
    capture_output=True,
    text=True,
    check=True,
)
print(show.stdout.strip())

## Reload the manifest in-process

[`Manifest.load`][gmat_sweep.Manifest.load] returns the full record — header fingerprint plus one [`ManifestEntry`][gmat_sweep.ManifestEntry] per completed run, in the order they finished.

In [ ]:
manifest = Manifest.load(manifest_path)
print(f"Header.run_count (planned):  {manifest.run_count}")
print(f"Entries on disk (completed): {len(manifest.entries)}")
print(f"GMAT install version:        {manifest.gmat_install_version}")
print(f"Script SHA-256 (canonical):  {manifest.script_sha256[:16]}...")

for entry in manifest.entries[:3]:
    print(f"  run_id={entry.run_id} status={entry.status} overrides={entry.overrides}")

## Aggregate the partial DataFrame

[`lazy_multiindex`][gmat_sweep.lazy_multiindex] is the same aggregator [`sweep()`][gmat_sweep.sweep] uses internally to produce its return value. Pointing it at a partial manifest yields a partial DataFrame — only the runs that completed before the kill are represented, but every one of them is intact.

In [ ]:
df = lazy_multiindex(manifest, out_dir)
print(f"Partial DataFrame shape: {df.shape}")
print(f"Distinct run_ids:        {df.index.get_level_values('run_id').nunique()}")
df.head()

## Where resume will land

Everything above uses **only the current public surface**:

- The manifest is fsync'd line-by-line so a kill leaves a parseable file.
- The CLI `show` subcommand reports the partial state.
- [`Manifest.load`][gmat_sweep.Manifest.load] tolerates a single torn final line.
- [`lazy_multiindex`][gmat_sweep.lazy_multiindex] aggregates whatever the manifest references; missing runs are simply absent.

**v0.2 will add `Sweep.from_manifest(manifest_path).resume()`** — programmatic re-dispatch of just the missing `run_id`s against the same backend, with the manifest header re-validated against the current script and environment. Until then, recovering a killed sweep is a re-issue of the original [`sweep()`][gmat_sweep.sweep] call: the durable manifest is the artefact you keep; the run command is the operation you re-run.

## Where to next

- **Manifest schema.** [Manifest schema](../manifest-schema.md) documents every field the JSON Lines records carry, plus the canonical script-hash convention.
- **CLI reference.** The [`gmat-sweep run` and `gmat-sweep show`](../parameter-spec.md) flags.